# ModelMaker3D Cloud (Hunyuan3D-2 on Colab Pro)

Mac の `make3d.sh` から使う高品質3D生成サーバー。形状+本物UVテクスチャ(paint)を A100 で生成する。

## 使い方(1ステップ)
1. **ランタイム → すべてのセルを実行**(A100+ハイメモリはノートブック側で設定済み。再起動も不要)
2. 最後のセルに「★ 固定URL起動OK」と出たら完了。**URLを貼る必要はない**(MacのClaudeが自動検出)。セルは動かしたままにする
   - 固定URLが取れなかった時だけ、表示される予備URL(trycloudflare)を Claude に貼る

うまくいかない時は「ランタイム → ランタイムを接続解除して削除」で真っさらにしてから 1. をやり直す。
セットアップ中に pip の依存関係エラー(cudf/cuml/pytensor等)が赤く出るが、使わないライブラリなので無視してよい。


In [ ]:
# 0) GPU 確認(torchはまだimportしない: setup後の再起動を不要にするため)
import subprocess
r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert r.returncode == 0 and 'GPU' in r.stdout, 'GPUランタイムを選択してください(ランタイム→タイプを変更→GPU)'
print(r.stdout.strip())


In [ ]:
# 1) セットアップ(初回のみ。2回目以降は自動でスキップされる)
# 注意: このセルより前に numpy/torch を import しないこと(再起動が必要になる)
import os
DONE = '/content/.mm3d_setup_done'
if os.path.exists(DONE):
    print('セットアップ済み → スキップ。このまま下のセルが実行されればOK')
else:
    if not os.path.exists('/content/Hunyuan3D-2'):
        !git clone https://github.com/Tencent/Hunyuan3D-2.git /content/Hunyuan3D-2
    os.chdir('/content/Hunyuan3D-2')
    !pip -q install -r requirements.txt
    !pip -q install -e .
    !pip -q install fastapi 'uvicorn[standard]' python-multipart rembg trimesh
    # paint(本物UVテクスチャ)用コンポーネントのビルド
    import subprocess
    for d in ('hy3dgen/texgen/custom_rasterizer', 'hy3dgen/texgen/differentiable_renderer'):
        try:
            print('building', d)
            subprocess.run('pip -q install -e .', shell=True, cwd='/content/Hunyuan3D-2/'+d, check=True)
        except Exception as e:
            print('build failed(形状のみになる可能性):', d, e)
    # paint読み込みバグ修正: DiffusionPipeline に trust_remote_code=True が必須
    mv = '/content/Hunyuan3D-2/hy3dgen/texgen/utils/multiview_utils.py'
    src = open(mv).read()
    if 'trust_remote_code' not in src:
        open(mv, 'w').write(src.replace('torch_dtype=torch.float16', 'torch_dtype=torch.float16, trust_remote_code=True', 1))
        print('patched: trust_remote_code')
    # requirements が numpy を古い版に下げ、Colab本体のライブラリと衝突する。
    # 最後に 2.3系へ戻して統一する(2026-07-01 に動作実績のある構成)。
    !pip -q install --force-reinstall --no-cache-dir "numpy>=2.3,<2.4" pillow
    open(DONE, 'w').write('ok')
    print('セットアップ完了。このまま下のセルへ進む(再起動不要)。')

# --- 整合性チェック(毎回実行・数秒) ---
# requirements が Pillow/numpy を部分的に入れ替えてファイル混在で壊れることがある
# (例: PIL.ImageText が新版、PIL._typing が旧版 → ImportError: _Ink)。検出したら強制再インストールで修復。
import subprocess as _sp
def _imports_ok(code):
    return _sp.run(['python', '-c', code], capture_output=True).returncode == 0
if not _imports_ok('import PIL.ImageText, PIL._typing'):
    print('Pillow の不整合を検出 → 修復中...')
    _sp.run('pip -q install --force-reinstall --no-cache-dir pillow', shell=True)
    assert _imports_ok('import PIL.ImageText'), 'Pillow修復に失敗。ランタイムを接続解除して削除→やり直してください'
    print('Pillow 修復完了')
if not _imports_ok('import numpy'):
    print('numpy の不整合を検出 → 修復中...')
    _sp.run('pip -q install --force-reinstall --no-cache-dir "numpy>=2.3,<2.4"', shell=True)
print('ライブラリ整合性 OK')


In [ ]:
# 3) cloudflared を取得(公開URL用)
import os
if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
print('cloudflared ready')

In [ ]:
# 4) モデルを読み込み、FastAPI サーバを起動 → cloudflared で公開URLを表示
#    このセルは実行したままにしておくこと(サーバが動き続ける)。
import io, os, re, time, tempfile, threading, subprocess, uuid
import torch, trimesh
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import Response
import uvicorn

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.rembg import BackgroundRemover

# メッシュ後処理(あれば使う): 浮遊/縮退面の除去でメッシュ品質を底上げ
try:
    from hy3dgen.shapegen import FloaterRemover, DegenerateFaceRemover
    _floater, _degen = FloaterRemover(), DegenerateFaceRemover()
except Exception as e:
    print('mesh cleaners unavailable:', e); _floater = _degen = None

# 標準モデル。既定サブフォルダ(hunyuan3d-dit-v2-0)で確実に読める。
# 軽量版: from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini')
MODEL_ID = 'tencent/Hunyuan3D-2'
print('loading shape pipeline...', MODEL_ID)
shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(MODEL_ID)
try:
    shape_pipe.enable_flashvdm()  # あれば省メモリ/高速化
except Exception:
    pass
rembg = BackgroundRemover()

# paint(本物UVテクスチャ)を有効化。L4/A100(Colab Pro)推奨。
# T4等で載らない/OOMの場合は自動で『形状のみ』にフォールバック(クラッシュしない)。
ENABLE_PAINT = True
# paint テクスチャ解像度。L4=2048推奨、A100なら4096も可。
PAINT_TEX_RES = 4096
paint_pipe = None
if ENABLE_PAINT:
    # 新しい diffusers は Hunyuan の custom pipeline 読込に trust_remote_code=True が必須。
    # DiffusionPipeline.from_pretrained に既定で渡すようパッチ(paint 内部で使われる)。
    try:
        from diffusers import DiffusionPipeline as _DP
        if not getattr(_DP.from_pretrained, '_trc', False):
            _o = _DP.from_pretrained
            def _p(*a, **k):
                k.setdefault('trust_remote_code', True)
                return _o(*a, **k)
            _p._trc = True
            _DP.from_pretrained = _p
    except Exception as _e:
        print('trust_remote_code patch skip:', _e)
    try:
        from hy3dgen.texgen import Hunyuan3DPaintPipeline
        paint_pipe = Hunyuan3DPaintPipeline.from_pretrained(MODEL_ID)
        cfg = getattr(paint_pipe, 'config', None)
        if cfg is not None:
            for attr in ('render_size', 'texture_size', 'tex_resolution'):
                if hasattr(cfg, attr):
                    try: setattr(cfg, attr, PAINT_TEX_RES)
                    except Exception: pass
        print('paint pipeline ready (tex_res target=%d)' % PAINT_TEX_RES)
    except Exception as e:
        print('paint unavailable (形状のみで返す):', e)

app = FastAPI()

@app.get('/health')
def health():
    return {'ok': True, 'paint': paint_pipe is not None}

def _run_shape(img, steps, octree):
    # octree_resolution 対応版 → 非対応版 の順で安全に呼ぶ
    for kw in (dict(num_inference_steps=steps, octree_resolution=octree),
               dict(num_inference_steps=steps)):
        try:
            return shape_pipe(image=img, **kw)[0]
        except TypeError:
            continue
    return shape_pipe(image=img)[0]

def _generate(raw: bytes, texture: str, steps: str, octree: str):
    """画像バイト列 → (glbバイト列, textured)。GPUは同時1ジョブ(_gpu_lock)。"""
    img = Image.open(io.BytesIO(raw)).convert('RGBA')
    img = rembg(img)
    try:
        octv = int(octree)
    except Exception:
        octv = 384
    with _gpu_lock:
        mesh = _run_shape(img, int(steps), octv)
        # メッシュ後処理(あれば)
        for fn in (_floater, _degen):
            if fn is not None:
                try: mesh = fn(mesh)
                except Exception as e: print('cleaner skip:', e)
        textured = False
        if texture != 'shape' and paint_pipe is not None:
            try:
                mesh = paint_pipe(mesh, image=img)
                textured = True
            except Exception as e:
                print('paint failed, returning shape-only:', e)
                torch.cuda.empty_cache()
        path = tempfile.mktemp(suffix='.glb')
        mesh.export(path)
        data = open(path, 'rb').read()
        torch.cuda.empty_cache()
    return data, textured

_gpu_lock = threading.Lock()
_jobs = {}  # jid -> {'status': 'running'|'done'|'error', 'data':..., 'textured':..., 'error':...}

def _job_worker(jid, raw, texture, steps, octree):
    try:
        data, textured = _generate(raw, texture, steps, octree)
        _jobs[jid] = {'status': 'done', 'data': data, 'textured': textured}
    except Exception as e:
        import traceback; traceback.print_exc()
        _jobs[jid] = {'status': 'error', 'error': str(e)}

# 非同期ジョブAPI(推奨)。trycloudflareは約100秒でタイムアウト(524)するため、
# 長時間の生成は「投げる→ポーリング→取得」で行う。
@app.post('/gen_async')
def gen_async(image: UploadFile = File(...), texture: str = Form('auto'),
              steps: str = Form('50'), octree: str = Form('384')):
    raw = image.file.read()
    jid = uuid.uuid4().hex[:12]
    _jobs[jid] = {'status': 'running'}
    threading.Thread(target=_job_worker, args=(jid, raw, texture, steps, octree), daemon=True).start()
    return {'job': jid}

@app.get('/status/{jid}')
def status(jid: str):
    j = _jobs.get(jid)
    if j is None:
        return {'status': 'unknown'}
    return {'status': j['status'], 'error': j.get('error')}

@app.get('/result/{jid}')
def result(jid: str):
    j = _jobs.get(jid)
    if j is None or j['status'] != 'done':
        return Response(status_code=404)
    return Response(content=j['data'], media_type='model/gltf-binary',
                    headers={'X-Textured': '1' if j['textured'] else '0'})

# 同期API(互換用)。100秒未満で終わる場合のみ有効。
@app.post('/gen')
def gen(image: UploadFile = File(...), texture: str = Form('auto'),
        steps: str = Form('50'), octree: str = Form('384')):
    raw = image.file.read()
    data, textured = _generate(raw, texture, steps, octree)
    return Response(content=data, media_type='model/gltf-binary', headers={'X-Textured': '1' if textured else '0'})

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

# 固定URLトンネル(localtunnel): MacのClaudeがURLを自動検出する。貼り付け不要。
LT_SUBDOMAIN = 'mm3d-getabako'
lt_url = None
try:
    lt = subprocess.Popen(['npx', '--yes', 'localtunnel', '--port', '8000', '--subdomain', LT_SUBDOMAIN],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(60):
        line = lt.stdout.readline()
        if not line:
            break
        m = re.search(r'https://[-a-z0-9]+\.loca\.lt', line)
        if m:
            lt_url = m.group(0)
            break
except Exception as e:
    print('localtunnel skip:', e)

# 予備トンネル(cloudflared): 固定URLが取れなかった時の手貼り用
proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

print('\n' + '='*60)
if lt_url == f'https://{LT_SUBDOMAIN}.loca.lt':
    print('  ★ 固定URL起動OK:', lt_url)
    print('  MacのClaudeが自動検出するので、URLを貼る必要はありません。')
    print('  そのまま「3Dにして」と頼めばOK。')
else:
    print('  固定URLが取れませんでした。予備URLを Claude に貼ってください:')
    print('   ', url)
print('  (予備URL: %s)' % url)
print('='*60)
for line in proc.stdout:
    pass